Name of Machine Learning Model: Linear SVM

In [1]:
%pip install pandas
%pip install scikit-learn
%pip install kagglehub
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



In [2]:
import kagglehub
kagglehub.login()

In [3]:
# Loading the data 
from pathlib import Path
import kagglehub
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

COMPETITION_NAME = "50-007-machine-learning-may-2026"

competition_path = Path(kagglehub.competition_download(COMPETITION_NAME))

train_df = pd.read_csv(competition_path / "train_features.csv")
test_df = pd.read_csv(competition_path / "test_features.csv")

# Split data for training and validation
X = train_df.drop(columns=['label', 'id'])
y = train_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print( X_train,'\n', X_test)
print( y_train,'\n', y_test)

           0001      0002      0003      0004      0005      0006  0007  \
1443   0.000000  0.000000  0.029357  0.000000  0.079082  0.000000   0.0   
13294  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.0   
5358   0.000000  0.075642  0.000000  0.081136  0.087138  0.082358   0.0   
7041   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.0   
7553   0.000000  0.000000  0.000000  0.000000  0.063735  0.060239   0.0   
...         ...       ...       ...       ...       ...       ...   ...   
5341   0.068407  0.000000  0.000000  0.000000  0.000000  0.000000   0.0   
14113  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.0   
2786   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.0   
14348  0.000000  0.000000  0.000000  0.037527  0.000000  0.117302   0.0   
6134   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   0.0   

           0008  0009      0010  ...  4991     4992  4993  4994  4995  4996  \
1443   0.000000   0.

In [4]:
# Check data distribution between the 2 classes
print(y.value_counts(normalize=True))

label
1    0.6252
0    0.3748
Name: proportion, dtype: float64


In [5]:
def train_custom_svm(X, y, learning_rate=0.1, lambda_param=0.01, epochs=1000, verbose=False):
    X = np.array(X, dtype=np.float64)
    y_ = np.where(np.array(y) == 0, -1, 1)

    n_samples, n_features = X.shape

    n_human = np.sum(y_ == -1)
    n_machine = np.sum(y_ == 1)
    weight_human = n_samples / (2 * n_human)
    weight_machine = n_samples / (2 * n_machine)
    sample_weights = np.where(y_ == -1, weight_human, weight_machine)
    w = np.zeros(n_features)
    b = 0.0

    for epoch in range(epochs):
        scores = X @ w + b
        margins = y_ * scores

        mask = (margins < 1).astype(np.float64)          # 1.0 where violated, 0.0 otherwise
        weighted_term = mask * sample_weights * y_        # combine mask + class weight + label, all in one vector

        dw = lambda_param * w - (X.T @ weighted_term) / n_samples   # single matrix-vector multiply, no copying
        db = -np.sum(weighted_term) / n_samples

        w -= learning_rate * dw
        b -= learning_rate * db

        if verbose and epoch % 100 == 0:
            hinge_loss = np.sum(sample_weights * np.maximum(0, 1 - margins)) / n_samples
            reg_loss = 0.5 * lambda_param * np.dot(w, w)
            print(f"Epoch {epoch}: loss={hinge_loss + reg_loss:.4f}, violated={int(mask.sum())}")

    return w, b

def predict_custom_svm(X, w, b):
    X = np.array(X, dtype=np.float64)
    scores = X @ w + b
    return np.where(scores >= 0, 1, 0)

In [6]:
# F1 Calculations functions

def f1_calculate(predictions, y_test):
    # F1 calc for human +ve
    TP_0 = 0
    FP_0 = 0
    FN_0 = 0
    for i in range(len(predictions)):
        if(predictions[i] == 0):
            if(y_test.iloc[i] == predictions[i]):
                TP_0 += 1
            else:
                FP_0 += 1
        else:
            if(y_test.iloc[i] != predictions[i]):
                FN_0 += 1

    F1_0 = TP_0 / (TP_0 + 0.5*(FP_0 + FN_0))
    #print(F1_0)  

    # F1 calc for machine +ve
    TP_1 = 0
    FP_1 = 0
    FN_1 = 0
    for i in range(len(predictions)):
        if(predictions[i] == 1):
            if(y_test.iloc[i] == predictions[i]):
                TP_1 += 1
            else:
                FP_1 += 1
        else:
            if(y_test.iloc[i] != predictions[i]):
                FN_1 += 1

    F1_1 = TP_1 / (TP_1 + 0.5*(FP_1 + FN_1))
    # print(F1_1)  

    macro_F1 = (F1_0 + F1_1) / 2
    # print(macro_F1)  
    print(macro_F1, F1_0, F1_1)
    return (macro_F1)

#print(f1_calculate(predictions, y_test))

In [7]:
#Train model and calculate f1 score:

w, b = train_custom_svm(X_train, y_train, 1.0, 0.01, 1000, True)

predictions = predict_custom_svm(X_test, w, b)

f1_macro = f1_calculate(predictions, y_test)
print(f1_macro)

Epoch 0: loss=1.0000, violated=14000
Epoch 100: loss=0.9810, violated=14000
Epoch 200: loss=0.9783, violated=14000
Epoch 300: loss=0.9780, violated=14000
Epoch 400: loss=0.9779, violated=14000
Epoch 500: loss=0.9779, violated=14000
Epoch 600: loss=0.9779, violated=14000
Epoch 700: loss=0.9779, violated=14000
Epoch 800: loss=0.9779, violated=14000
Epoch 900: loss=0.9779, violated=14000
0.6354580681549734 0.6175955134945671 0.6533206228153797
0.6354580681549734


In [8]:
# try different lambda values

for lam in [0.0001, 0.001, 0.01, 0.1, 1]:
    w, b = train_custom_svm(X_train, y_train, learning_rate=1.0, lambda_param=lam, epochs=1000, verbose=False)
    predictions = predict_custom_svm(X_test, w, b)
    f1 = f1_calculate(predictions, y_test)
    print(f"lambda={lam}, f1={f1}")

0.6368304178886325 0.6358014374059836 0.6378593983712814
lambda=0.0001, f1=0.6368304178886325
0.6005459428481833 0.6246463376296763 0.5764455480666903
lambda=0.001, f1=0.6005459428481833
0.6354580681549734 0.6175955134945671 0.6533206228153797
lambda=0.01, f1=0.6354580681549734
0.6354580681549734 0.6175955134945671 0.6533206228153797
lambda=0.1, f1=0.6354580681549734
0.6354580681549734 0.6175955134945671 0.6533206228153797
lambda=1, f1=0.6354580681549734


In [9]:
# scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
#find best parameters

best = 0

for lr in [0.3,0.1,0.03,0.01]:
    for lam in [0.00001,0.0001,0.001,0.01,0.1]:
        w,b = train_custom_svm(
            X_train_scaled,
            y_train,
            learning_rate=lr,
            lambda_param=lam,
            epochs=1000
        )

        pred = predict_custom_svm(X_test_scaled,w,b)
        f1 = f1_calculate(pred,y_test)

        if f1 > best:
            best = f1
            print(best, lr, lam)

0.6765303748995327 0.6010002174385736 0.7520605323604919
0.6765303748995327 0.3 1e-05
0.6769452335858586 0.6019965277777778 0.7518939393939394
0.6769452335858586 0.3 0.0001
0.6741696252254622 0.5983054529654573 0.7500337974854671
0.6843991854693645 0.6142306043720531 0.7545677665666758
0.6843991854693645 0.3 0.01
0.7099798501363233 0.6436135725091853 0.7763461277634612
0.7099798501363233 0.3 0.1
0.6792828213247107 0.6064265689023075 0.752139073747114
0.6775722491322228 0.6040587219343696 0.751085776330076
0.6772832596906055 0.6049409237379162 0.7496255956432948
0.6837436805314742 0.613903743315508 0.7535836177474403
0.703899552113155 0.640830156713257 0.766968947513053
0.6886425996463345 0.6202477573686459 0.757037441924023
0.6887382414655983 0.6205719163465643 0.7569045665846322
0.6877185256351152 0.6201616333475117 0.7552754179227186
0.6904114518045128 0.6235919234856535 0.7572309801233722
0.7033100959149272 0.6415491475478846 0.7650710442819699
0.6984973259915846 0.6362872742545149 

In [12]:
# train model with the best hyperparameters identified
w, b = train_custom_svm(
    X_train_scaled,
    y_train,
    learning_rate=0.3,
    lambda_param=0.1,
    epochs=1000
)


# Use the model on the test data for kaggle submission

# Prepare test features
test_X = test_df.drop(columns=['id'])

# Apply the SAME scaler fitted on the training set
test_X_scaled = scaler.transform(test_X)

# Predict
test_predictions = predict_custom_svm(test_X_scaled, w, b)

submission_df = pd.DataFrame({
    "id": test_df["id"],
    "label": test_predictions
})

submission_df.to_csv("output.csv", index=False)

In [13]:
# check if value of weight and bias is proportionate
print(pd.Series(predictions).value_counts())

print(w[:20])   #first 20 weights
print(b)
print(np.all(w == 0))

0    3457
1    2543
Name: count, dtype: int64The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.

[-0.02341883 -0.00970362 -0.00153874 -0.05170081 -0.01114861  0.04934495
 -0.01120896 -0.05556132 -0.00617603  0.00460911  0.01836699  0.01458355
  0.00690286  0.02126503  0.06960427 -0.00046365 -0.00766751 -0.01053711
 -0.0131306   0.0235702 ]
0.43114287957597697
False
